# Experiments 17 and 18 — six multimodal LLM raters on still frames (17: step-25 recordings, 18: step-23 recordings)

# Environment

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../env/configs"))
from configs import *
list_users = list(range(1, 120))
group_names = {1: "ELVT", 2: "SYMP", 3: "INSP", 4: "CTRL"}

import re, json, time, base64, threading, traceback
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import psychai
print("psychai loaded from:", os.path.dirname(psychai.__file__), "| paper:", path_paper)
assert "psychai_20290921" in psychai.__file__, "an older psychai checkout was imported: restart the kernel (configs.py points to psychai_20290921)"
for _name in ("GEMINI_2_5_FLASH", "GEMINI_3_8_FLASH", "GPT_6_LUNA", "CLAUDE_HAIKU_4_5", "DEEPSEEK_V4_1_FLASH", "ALI_D_QWEN_3_8_FLASH"):
    assert hasattr(LLMModel, _name), f"LLMModel.{_name} missing: the psychai checkout is not the one with the six raters"
print("default llm_client of configs (not used by these experiments):", getattr(llm_client, "model_id", type(llm_client).__name__))


# Configuration

In [ ]:
# === Configuration ===

# The six multimodal LLM raters: the five of problematic_short_film/cache_friendly/exp_with_frame.ipynb plus Gemini 2.5 Flash (Experiment 1's model). key -> LLMModel. Every model rates every participant; a model whose client cannot be created
# (missing API key, unavailable model) is skipped with a warning. All models receive the same prompt and the same labelled frame layout.
llm_models = {
    "Gemini 2.5 Flash": LLMModel.GEMINI_2_5_FLASH,     # the Gemini of Experiment 1 (configs' default client)
    "Gemini 3.8 Flash":   LLMModel.GEMINI_3_8_FLASH,
    "GPT6 Luna":      LLMModel.GPT_6_LUNA,
    "Claude Haiku 4.5":   LLMModel.CLAUDE_HAIKU_4_5,      # Anthropic; needs api_key_claude in .env (or ANTHROPIC_API_KEY). Other ids: CLAUDE_FABLE_5_1, CLAUDE_SONNET_5, CLAUDE_HAIKU_4_5
    "Deepseek 4.1 Flash": LLMModel.DEEPSEEK_V4_1_FLASH,
    "Qwen 3.8 Flash":     LLMModel.ALI_D_QWEN_3_8_FLASH,   # Aliyun DashScope (OpenAI-compatible); key = api_key_aliyun_domestic
}
claude_max_tokens = 16000          # output cap for the Claude models (non-streaming request), as in exp_with_frame
llm_temperature = 0
llm_seed = 42

# Frame sampling: n_frames_per_video frames per recording, taken at 1/(n+1), 2/(n+1), ..., n/(n+1) of its total frame count (for 6 frames:
# 1/7 ... 6/7), i.e. at equal spacing across the recording, in time order; long side frame_max_side px, JPEG quality frame_jpeg_quality;
# cached in path_frames_dir.
n_frames_per_video = 8
frame_positions = [k / (n_frames_per_video + 1) for k in range(1, n_frames_per_video + 1)]     # fractions of the total frame count
frame_max_side = 640
frame_jpeg_quality = 85
frame_count_check = None                 # how a response's frame count is verified (as in exp_with_frame): "self_report" = frames_analyzed must equal the
                                   # number of images sent; "tokens" = prompt_tokens must match the per-model token accounting (a + b * frames,
                                   # fitted on responses that counted right); "either" = one of the two suffices; None = no check. Some models
                                   # miscount the images although every image arrived; "either" keeps those responses instead of retrying them.
frame_token_tolerance = 0.10       # tokens rule: |prompt_tokens - expected| / expected <= tolerance

# Format retry: a response without parseable scores is requested once more with a short reminder appended (0 disables it)
max_parse_retries = 1
parse_retry_reminder = """

=== Reminder ===

Your previous answer did not follow the output format. Output the strict format: (sympathy, inspiration) = (X, Y) with integers from 1 to 10, then the explanation."""

# Experiment control
run_mode = "full"                  # "pilot" = the first n_pilot_participants participants; "full" = all
n_pilot_participants = 24
run_experiments = True              # False = only analyse what the checkpoint holds
overwrite_existing = False          # True = back up the checkpoint and re-run everything
max_workers = 12
cache_warmup = True                # the first call of every model runs alone (it writes the prompt into the provider's cache), then the rest in parallel
prompt_version = "2026-09-26-exp17-v3"

# Paths shared by the experiments
path_compressed_video = os.path.join(path_paper, "temp", "compressed_video")
os.makedirs(os.path.join(path_paper, "temp", "pickle"), exist_ok=True)
# The frame experiments: experiment number -> recording step. The video table is merged_video_<step>.plk (the step-25 and step-23 tables), the results pickle df_<step>_exp_<n>.plk, the frames temp/frames_<n>/, the outputs experiment_<n>*.csv.
experiments_frames = {17: 25, 18: 23}
print(f"models = {list(llm_models)}; run_mode = {run_mode}; {n_frames_per_video} frames per recording at {[f'{k}/{n_frames_per_video + 1}' for k in range(1, n_frames_per_video + 1)]} of its frames; frame experiments (number -> recording step) = {experiments_frames}")


# Prompt of Experiments 17 and 18 (Experiment 1 of experiments_1_9: shared template code, attachment described as frames) and select_experiment()

In [ ]:
# === Prompt of Experiments 17 and 18: the prompt of Experiment 1 of experiments_1_9.ipynb, shared code ===
# Experiment 1 composes template_main_25 = part_first_part_with_video + part_main + last_part. The template cell of experiments_1_9.ipynb
# (same folder) is executed here, and only the three parts that this composition uses are taken from it — nothing is copied, nothing
# unused is kept. The frames change exactly two things: the first sentence of part_first (n_frames_per_video still frames at equal spacing
# across the recording instead of "the first attached video") and the hard constraint of last_part (no image instead of no video file); part_main and
# the rest of the two parts are Experiment 1's own text. [[COT]] / [[KNOWLEDGE]] / [[TOOLS]] are resolved (to nothing, as in Experiment 1)
# by experiment_helper.get_prompt_text_template in select_experiment().
path_source_notebook = os.path.join(os.getcwd(), "experiments_1_9.ipynb")
_cells = json.load(open(path_source_notebook, encoding="utf-8"))["cells"]
_template_cells = [c for c in _cells if c["cell_type"] == "code" and "template_main_25 = part_first_part_with_video + part_main + last_part" in "".join(c["source"])]
assert len(_template_cells) == 1, f"template cell of Experiment 1 not found in {path_source_notebook}"
_ns = {}
exec("".join(_template_cells[0]["source"]), _ns)
part_first_part_with_video, part_main, last_part = _ns["part_first_part_with_video"], _ns["part_main"], _ns["last_part"]

def _replace_once(text, old, new):
    assert text.count(old) == 1, f"template text changed; expected exactly once: {old[:60]!r}"
    return text.replace(old, new)

part_first_part_with_frames = _replace_once(part_first_part_with_video,
    "The first attached video is a behavioral recording captured during an experimental procedure in which an individual is watching a short film.",
    f"The attached images are {n_frames_per_video} still frames of a behavioral recording, taken in time order at equal intervals across its duration "
    f"(at {', '.join(f'{k}/{n_frames_per_video + 1}' for k in range(1, n_frames_per_video + 1))} of the recording), captured during an experimental "
    f"procedure in which an individual is watching a short film.")
last_part_frames = _replace_once(last_part, "* If no video file is attached, return exactly:", "* If no image is attached, return exactly:")
# Addition to part_first, per experiment (both experiments use the same sentence): the frames follow the film in sequence and their
# change over time, together with the images, is the evidence for being moved (sympathy) or inspired. Composed in select_experiment().
_sequence_sentence = ("These frames were extracted in sequence while the individual was watching the film. Observe how the individual's facial expression, "
                      "gaze and posture change over time, and use the images and these changes to judge whether the individual was moved (sympathy) or "
                      "inspired (inspiration) by the film.")
frame_sequence_note = {17: _sequence_sentence, 18: _sequence_sentence}
def compose_template(exp_no):
    """Experiment 1's composition (part_first + part_main + last_part) for one frame experiment: the frames sentence, the experiment's
    sequence note (if any) as a paragraph of part_first, then part_main and last_part unchanged."""
    note = frame_sequence_note.get(exp_no, "")
    part_first = part_first_part_with_frames.rstrip("\n") + ("\n\n" + note if note else "") + "\n\n"
    return part_first + part_main + last_part_frames
template_main_frames_raw = compose_template(17)      # the composition (as template_main_25 in Experiment 1) with the sequence note
print(f"prompt = Experiment 1's template_main_25 composition with the attachment described as {n_frames_per_video} frames at equal spacing ({len(template_main_frames_raw):,} chars); sequence note for experiments {sorted(frame_sequence_note)}")


def select_experiment(exp_no):
    """Activate one frame experiment (17 = step-25 recordings, 18 = step-23 recordings): its video table, results pickle, frames folder,
    output files and prompt version; the experiment definition (experiment_helper.copy() + define_experiment, as experiments_1_9's
    Experiment 1, with experiment_step = 2 as there); Experiment 1's prompt (attachment described as frames) resolved by get_prompt_text_template; and the participants
    from experiment_helper.get_df_video_path()."""
    global exp_no_active, exp_label, experiment_step, path_pickle_df_video, path_checkpoint_pickle, path_frames_dir, path_output_csv, prompt_version
    global experiment_helper_15, template_main_15, df_video, df_participants, all_participants, participants, missing
    step = experiments_frames[exp_no]
    exp_no_active, exp_label, experiment_step = exp_no, f"Exp {exp_no}", step
    path_pickle_df_video = os.path.join(path_paper, "temp", "pickle", f"merged_video_{step}.plk")
    path_checkpoint_pickle = os.path.join(path_paper, "temp", "pickle", f"df_{step}_exp_{exp_no}.plk")     # path_pickle_df_experiment_results
    path_frames_dir = os.path.join(path_paper, "temp", f"frames_{exp_no}_{n_frames_per_video}pos"); os.makedirs(path_frames_dir, exist_ok=True)
    path_output_csv = os.path.join(os.getcwd(), f"experiment_{exp_no}.csv")
    prompt_version = f"2026-09-26-exp{exp_no}-v5-{n_frames_per_video}frames" + ("-seq" if frame_sequence_note.get(exp_no) else "")   # v5: template_main_25 with n frames at equal spacing; -seq = with the sequence note
    template_raw = compose_template(exp_no)
    helper = experiment_helper.copy()
    helper.define_experiment(
        participant_sn_range=list_users,
        path_pickle_df_video=path_pickle_df_video,
        path_pickle_df_experiment_results=path_checkpoint_pickle,
        path_compressed_video=path_compressed_video,
        experiment_step=2,
        video_only=True,
        template_main=template_raw,
        interpreters=[],
        template_cot="",
        template_knowledge_injection="",
        template_tools="",
    )
    template_main_15 = helper.get_prompt_text_template(template_experiment=template_raw, template_cot="", template_knowledge_injection="", template_tools="")
    experiment_helper_15 = helper
    df_video = helper.get_df_video_path()
    df_participants = df_video.loc[:, ~df_video.columns.duplicated()].copy()
    df_participants["Participant SN"] = df_participants["Participant SN"].astype(str).str.zfill(3)
    df_participants["Group"] = pd.to_numeric(df_participants["Group"], errors="coerce")
    df_participants["Group Name"] = df_participants["Group"].map(group_names)
    df_participants["Compressed Video Path"] = df_participants["Compressed Video Path"].astype(str)
    df_participants["video_ok"] = df_participants["Compressed Video Path"].map(os.path.isfile)
    df_participants = df_participants.drop_duplicates("Participant SN").sort_values("Participant SN").reset_index(drop=True)
    all_participants = df_participants.loc[df_participants["video_ok"], "Participant SN"].tolist()
    participants = all_participants[:n_pilot_participants] if run_mode == "pilot" else all_participants
    missing = df_participants.loc[~df_participants["video_ok"], "Participant SN"].tolist()
    print(f"=== {exp_label}: step-{step} recordings ({os.path.basename(path_pickle_df_video)}), checkpoint {os.path.basename(path_checkpoint_pickle)}, prompt_version {prompt_version} ===")
    print(f"{len(df_participants)} participants in the table; compressed video present for {len(all_participants)}" + (f"; MISSING for {missing}" if missing else ""))
    print("Groups:", df_participants["Group Name"].value_counts(dropna=False).to_dict())
    print(f"run_mode = {run_mode}: {len(participants)} participants -> {participants if run_mode == 'pilot' else '(all)'}")

print(template_main_frames_raw)


# Frames, requests and parsing

In [ ]:
# === Frames (n_frames_per_video per recording at equal spacing, cached), the cache-friendly labelled request for every provider, the frame-count check, and the response parser ===
import cv2
_frames_lock = threading.Lock()

def extract_frames(video_path, participant_sn):
    """Take n_frames_per_video frames of a recording at the frame indices round(N * p) for p in frame_positions (N = total frame count:
    1/7, 2/7, ... 6/7 of the recording for 6 frames), resize so that the long side is frame_max_side px, save as JPEG. Cached: a manifest
    next to the frames makes every later run reuse them. Returns [{t: seconds, path: jpeg, frame_index: i}, ...] in time order."""
    out_dir = os.path.join(path_frames_dir, participant_sn, os.path.splitext(os.path.basename(video_path))[0])
    manifest = os.path.join(out_dir, "manifest.json")
    sig = {"size": os.path.getsize(video_path), "mtime": os.path.getmtime(video_path), "positions": frame_positions, "max_side": frame_max_side}
    with _frames_lock:
        if os.path.exists(manifest):
            try:
                m = json.load(open(manifest, "r", encoding="utf-8"))
                if m.get("signature") == sig and all(os.path.exists(f["path"]) for f in m["frames"]):
                    return m["frames"]
            except Exception:
                pass
        os.makedirs(out_dir, exist_ok=True)
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"cannot open {video_path}")
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames = []
        for p in frame_positions:
            idx = min(max(int(round(n_total * p)), 0), max(n_total - 1, 0))
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, img = cap.read()
            if not ok:                                   # some containers seek imprecisely near the end: fall back to a time seek
                cap.set(cv2.CAP_PROP_POS_MSEC, idx / fps * 1000)
                ok, img = cap.read()
            if not ok:
                continue
            h, w = img.shape[:2]
            scale = frame_max_side / max(h, w)
            if scale < 1:
                img = cv2.resize(img, (int(round(w * scale)), int(round(h * scale))), interpolation=cv2.INTER_AREA)
            t = idx / fps
            mm, ss = divmod(int(round(t)), 60)
            path = os.path.join(out_dir, f"f{len(frames):03d}_{mm:02d}m{ss:02d}s.jpg")
            cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, frame_jpeg_quality])
            frames.append({"t": float(t), "path": path, "frame_index": idx})
        cap.release()
        json.dump({"signature": sig, "video": video_path, "fps": fps, "n_frames_total": n_total, "frames": frames}, open(manifest, "w", encoding="utf-8"), indent=1)
        return frames

def time_label(frame):
    mm, ss = divmod(int(round(frame["t"])), 60)
    return f"Frame at {mm:02d}:{ss:02d} of the recording:"

def _jpeg_bytes(path):
    with open(path, "rb") as f:
        return f.read()

def provider_kind(model_key):
    p = llm_models[model_key].provider
    if p in (LLMType.GEMINI, LLMType.GEMINI_VERTEX):
        return "gemini"
    if p == LLMType.ANTHROPIC:
        return "anthropic"
    return "openai"

from google.genai import types
generate_config = (types.GenerateContentConfig(temperature=0.0, top_p=0.0, seed=llm_seed, candidate_count=1) if llm_temperature == 0
                   else types.GenerateContentConfig(temperature=llm_temperature, candidate_count=1))

# --- prompt-cache helpers (as in problematic_short_film/cache_friendly): the providers cache the longest identical PREFIX of a request
#     (OpenAI, DeepSeek, Gemini implicit and DashScope implicit caching automatically; Anthropic at explicit cache_control breakpoints).
#     The request therefore starts with the prompt text, which is identical for every participant, and the frames follow it.
request_trailer = ("All frames of the recording are above, in time order. Now answer for this participant in the strict output format given "
                   "in the instructions.")

def cached_tokens_of(usage):
    """Input tokens served from the provider's prompt cache: Gemini cached_content_token_count, DeepSeek prompt_cache_hit_tokens,
    Anthropic cache_read_input_tokens, OpenAI / DashScope prompt_tokens_details.cached_tokens. None when nothing is reported."""
    if usage is None:
        return None
    for attr in ("cached_content_token_count", "prompt_cache_hit_tokens", "cache_read_input_tokens"):
        v = getattr(usage, attr, None)
        if v is None:
            v = (getattr(usage, "model_extra", None) or {}).get(attr)
        if v is not None:
            return int(v)
    det = getattr(usage, "prompt_tokens_details", None)
    v = getattr(det, "cached_tokens", None) if det is not None else None
    return int(v) if v is not None else None

def openai_create_with_fallbacks(openai_client, kwargs, max_retries=3):
    """chat.completions.create that drops, one at a time, the optional parameters a provider or model rejects — seed, prompt_cache_key,
    temperature (some GPT models accept only the default temperature) — and retries; any other error is raised."""
    for _ in range(max_retries):
        try:
            return openai_client.chat.completions.create(**kwargs)
        except Exception as e:
            msg = str(e).lower()
            if "seed" in kwargs and "seed" in msg:
                kwargs.pop("seed")
            elif "prompt_cache_key" in kwargs and "prompt_cache_key" in msg:
                kwargs.pop("prompt_cache_key")
            elif "temperature" in kwargs and "temperature" in msg:
                kwargs.pop("temperature")
            else:
                raise
    return openai_client.chat.completions.create(**kwargs)

def _frame_parts(frames, text_part, image_part):
    out = [text_part(f"Recording of the participant watching the short film: {len(frames)} frames taken at equal intervals across the recording, in time order.")]
    for fr in frames:
        out.append(text_part(time_label(fr)))
        out.append(image_part(fr))
    return out

def generate_with_frames(model_key, prompt, frames, trailer_extra=""):
    """Send the cache-friendly layout — the prompt text first, then the header and label + image per frame, then a short closing note —
    to one model. trailer_extra (e.g. the format reminder of a retry) is appended to the closing note so that the cached prefix stays
    unchanged. Returns (text, prompt_tokens, output_tokens, finish_reason, cached_tokens)."""
    client = llm_clients[model_key]; model_name = llm_models[model_key].model_name; kind = provider_kind(model_key)
    trailer = request_trailer + trailer_extra
    if kind == "gemini":
        parts = [types.Part(text=prompt)]
        parts += _frame_parts(frames, lambda t: types.Part(text=t), lambda fr: types.Part.from_bytes(data=_jpeg_bytes(fr["path"]), mime_type="image/jpeg"))
        parts.append(types.Part(text=trailer))
        raw = client.client.models.generate_content(model=model_name, contents=[types.Content(role="user", parts=parts)], config=generate_config)
        try:
            text = (raw.text or "").strip()
        except Exception as e:
            text = f"[No valid response from model: {e}]"
        um = getattr(raw, "usage_metadata", None); finish = ""
        try:
            finish = raw.candidates[0].finish_reason.name
        except Exception:
            pass
        return text, getattr(um, "prompt_token_count", None), getattr(um, "candidates_token_count", None), finish, cached_tokens_of(um)
    if kind == "anthropic":
        blocks = [{"type": "text", "text": prompt, "cache_control": {"type": "ephemeral"}}]     # the common prefix = the whole prompt
        blocks += _frame_parts(frames, lambda t: {"type": "text", "text": t},
                               lambda fr: {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": base64.b64encode(_jpeg_bytes(fr["path"])).decode("ascii")}})
        blocks.append({"type": "text", "text": trailer})
        kwargs = dict(model=model_name, max_tokens=claude_max_tokens, system="You are an intelligent assistant.", messages=[{"role": "user", "content": blocks}])
        if client._anthropic_accepts_sampling():
            kwargs["temperature"] = llm_temperature
        if client._anthropic_fallbacks_apply():
            try:
                raw = client.client.beta.messages.create(betas=["server-side-fallback-2026-07-01"], fallbacks="default", **kwargs)
            except TypeError:
                raw = client.client.beta.messages.create(extra_headers={"anthropic-beta": "server-side-fallback-2026-07-01"}, extra_body={"fallbacks": "default"}, **kwargs)
        else:
            raw = client.client.messages.create(**kwargs)
        text = "".join(b.text for b in raw.content if b.type == "text").strip()
        if raw.stop_reason == "refusal":
            text = f"[Claude refused the request: {getattr(getattr(raw, 'stop_details', None), 'category', None)}]\n" + text
        return text, raw.usage.input_tokens, raw.usage.output_tokens, str(raw.stop_reason), cached_tokens_of(raw.usage)
    content = [{"type": "text", "text": prompt}]
    content += _frame_parts(frames, lambda t: {"type": "text", "text": t},
                            lambda fr: {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + base64.b64encode(_jpeg_bytes(fr["path"])).decode("ascii")}})
    content.append({"type": "text", "text": trailer})
    kwargs = dict(model=model_name, messages=[{"role": "system", "content": "You are an intelligent assistant."}, {"role": "user", "content": content}],
                  temperature=llm_temperature, seed=llm_seed, stream=False)
    raw = openai_create_with_fallbacks(client.client, kwargs)
    choice = raw.choices[0]; usage = getattr(raw, "usage", None)
    return ((choice.message.content or "").strip(), getattr(usage, "prompt_tokens", None), getattr(usage, "completion_tokens", None),
            str(getattr(choice, "finish_reason", "")), cached_tokens_of(usage))

# --- frame-count verification by token accounting (as in exp_with_frame): some models report fewer frames than they received
def calibrate_frame_tokens(records):
    """Per model: least-squares fit prompt_tokens ≈ a + b · frames_expected over responses whose self-reported frame count was right
    (>= 2 distinct frame counts, >= 4 responses). Returns {model_key: (a, b)}."""
    cal = {}
    df = pd.DataFrame(records) if len(records) else pd.DataFrame()
    if not len(df) or "prompt_tokens" not in df.columns:
        return cal
    ok = df[df["prompt_tokens"].notna() & df["frames_expected"].notna() & (df["frames_reported"] == df["frames_expected"]) & df["sympathy"].notna()]
    for mk, d in ok.groupby("model_key"):
        x = d["frames_expected"].astype(float).to_numpy(); y = d["prompt_tokens"].astype(float).to_numpy()
        if len(d) >= 4 and len(set(x)) >= 2:
            b, a = np.polyfit(x, y, 1)
            if b > 0:
                cal[mk] = (float(a), float(b))
    return cal

def frames_ok_by_tokens(model_key, prompt_tokens, n_frames, calibration):
    """True / False when the token accounting can decide, None when the model is not calibrated or no token count is known."""
    if prompt_tokens is None or pd.isna(prompt_tokens) or model_key not in calibration:
        return None
    a, b = calibration[model_key]; expected = a + b * n_frames
    return expected > 0 and abs(float(prompt_tokens) - expected) / expected <= frame_token_tolerance

def frame_check_verdict(frames_reported, n_frames, prompt_tokens, model_key, calibration):
    """(passed, how): how = "self_report" / "tokens" / "failed" / "none" (no check configured)."""
    if frame_count_check is None:
        return True, "none"
    by_report = frames_reported == n_frames
    by_tokens = frames_ok_by_tokens(model_key, prompt_tokens, n_frames, calibration)
    if frame_count_check == "self_report":
        return by_report, "self_report" if by_report else "failed"
    if frame_count_check == "tokens":
        return bool(by_tokens), "tokens" if by_tokens else "failed"
    if by_report:
        return True, "self_report"
    if by_tokens:
        return True, "tokens"
    return False, "failed"

token_calibration = {}      # filled by the run cell from the checkpoint before the calls start

def parse_llm_response(raw_text):
    """(sympathy, inspiration, explanation) exactly as Experiment 1 reads them — experiment_helper.segment_llm_response on a one-row frame —
    plus frames_analyzed, which only this experiment asks for."""
    out = {"sympathy": None, "inspiration": None, "frames_reported": None, "explanation": ""}
    if raw_text is None:
        return out
    s = str(raw_text).replace("（", "(").replace("）", ")").replace("，", ",").replace("＝", "=").strip()
    m_frames = re.search(r"frames_analy[sz]ed\s*=\s*(\d+)", s, re.IGNORECASE)
    if m_frames:
        out["frames_reported"] = int(m_frames.group(1))
    seg = experiment_helper.segment_llm_response(pd.DataFrame({"LLM_Result": [s], "sympathy": [None], "inspiration": [None], "explanation": [None]}))
    sym, insp = seg["sympathy"].iloc[0], seg["inspiration"].iloc[0]
    if pd.notna(sym) and pd.notna(insp) and 1 <= int(sym) <= 10 and 1 <= int(insp) <= 10:
        out["sympathy"], out["inspiration"] = int(sym), int(insp)
        out["explanation"] = (seg["explanation"].iloc[0] or "") if "explanation" in seg.columns else ""
    return out

print(parse_llm_response("frames_analyzed = 24\n(sympathy, inspiration) = (4, 1)\n\nThe participant ..."))

# LLM clients

In [ ]:
# === One client per rater; a rater whose client cannot be created is skipped ===
llm_clients = {}
for model_key, m in llm_models.items():
    if m.provider == LLMType.ANTHROPIC and not (os.getenv("api_key_claude") or os.getenv("ANTHROPIC_API_KEY")):
        print(f"[SKIP] {model_key:8s} -> {m.model_name}: no Anthropic key (set api_key_claude in the .env file)")
        continue
    try:
        llm_clients[model_key] = LLMClient(llm_model=m, temperature=llm_temperature)
        print(f"[OK]   {model_key:8s} -> {m.model_name} ({m.provider.name})")
    except Exception as e:
        print(f"[SKIP] {model_key:8s} -> {m.model_name}: client could not be created ({e})")
active_models = list(llm_clients)
if not active_models:
    raise RuntimeError("No LLM client could be created — check the API keys in the .env file.")
print(f"Active raters: {active_models}; temperature = {llm_temperature}")

# Experiment 17 — six raters, still frames, step-25 recordings (video table `merged_video_25.plk`)

In [ ]:
select_experiment(17)


## Preview: frames and the complete prompt of one participant

In [ ]:
# === Preview ===
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
sample_sn = participants[0]
sample_video = df_participants.set_index("Participant SN").loc[sample_sn, "Compressed Video Path"]
sample_frames = extract_frames(sample_video, sample_sn)
print(f"Participant {sample_sn} ({df_participants.set_index('Participant SN').loc[sample_sn, 'Group Name']}): {len(sample_frames)} frames from {os.path.basename(sample_video)}")
n_cols = 8; n_rows = max(1, -(-len(sample_frames) // n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 1.6 * n_rows), squeeze=False)
for k, fr in enumerate(sample_frames):
    ax = axes[k // n_cols, k % n_cols]; ax.imshow(mpimg.imread(fr["path"])); ax.set_title(time_label(fr).replace("Frame at ", "").rstrip(":").split(" of")[0], fontsize=7); ax.axis("off")
for j in range(len(sample_frames), n_rows * n_cols):
    axes[j // n_cols, j % n_cols].axis("off")
plt.tight_layout(); plt.show()
print("\n" + "=" * 100 + f"\nFULL PROMPT (identical for every participant and every model; the frames follow it)\n" + "=" * 100)
print(template_main_15)

## Run: every participant × every model (checkpointed; re-run the cell to resume)

In [ ]:
# === Run (the experiment activated by select_experiment) ===
def make_empty_result(sn, model_key):
    row = df_participants.set_index("Participant SN").loc[sn]
    return {"Participant SN": sn, "Experiment Step": experiment_step, "Group": row["Group"], "Group Name": row["Group Name"],
            "Video Path": row["Video Path"], "Compressed Video Path": row["Compressed Video Path"], "model_key": model_key,
            "model": llm_models[model_key].model_name, "temperature": llm_temperature, "prompt_version": prompt_version,
            "frame_positions": str(frame_positions), "frames_expected": None, "frames_reported": None, "frame_count_ok": None,
            "prompt_tokens": None, "cached_tokens": None, "output_tokens": None, "finish_reason": "", "retries": 0, "frame_check": "",
            "sympathy": None, "inspiration": None, "explanation": "", "LLM_Result": "", "individual_prompt": template_main_15,
            "status": "pending", "error": ""}

def rate_one(sn, model_key):
    r = make_empty_result(sn, model_key)
    try:
        frames = extract_frames(r["Compressed Video Path"], sn)
        r["frames_expected"] = len(frames)
        t0 = time.time()
        text, ptok, otok, finish, ctok = generate_with_frames(model_key, template_main_15, frames)
        parsed = parse_llm_response(text)
        n_retries = 0
        while parsed["sympathy"] is None and n_retries < max_parse_retries:
            n_retries += 1
            print(f"  [RETRY {n_retries}] {sn} - {model_key}: no scores in the answer -> asking again with the reminder")
            text, ptok, otok, finish, ctok = generate_with_frames(model_key, template_main_15, frames, trailer_extra=parse_retry_reminder)
            parsed = parse_llm_response(text)
        passed, how = frame_check_verdict(parsed["frames_reported"], len(frames), ptok, model_key, token_calibration)
        r.update(retries=n_retries, LLM_Result=text, prompt_tokens=ptok, cached_tokens=ctok, output_tokens=otok, finish_reason=finish, sympathy=parsed["sympathy"],
                 inspiration=parsed["inspiration"], explanation=parsed["explanation"], frames_reported=parsed["frames_reported"],
                 frame_count_ok=passed, frame_check=how)
        if parsed["sympathy"] is None:
            r["status"] = "parse_failed"
        elif not passed:
            r["status"] = "frame_count_mismatch"
        else:
            r["status"] = "success"
        print(f"  [{'OK' if r['status'] == 'success' else r['status'].upper()}] {sn} ({r['Group Name']}) - {model_key}: (sympathy, inspiration) = "
              f"({parsed['sympathy']}, {parsed['inspiration']}), frames_analyzed={parsed['frames_reported']} (sent {len(frames)}, verified by {how}), "
              f"tokens={ptok} (cached {ctok}), {time.time() - t0:.0f}s")
    except Exception as e:
        r["status"], r["error"] = "error", traceback.format_exc()
        print(f"  [ERROR] {sn} - {model_key}: {e}")
    return r

def result_key(r):
    return f"{r['Participant SN']}_{r['model_key']}"

tasks = [(sn, mk) for sn in participants for mk in active_models]
if run_experiments:
    all_results = []
    if os.path.exists(path_checkpoint_pickle):
        if overwrite_existing:
            backup = path_checkpoint_pickle.replace(".plk", f"_backup_{time.strftime('%Y%m%d_%H%M%S')}.plk")
            os.replace(path_checkpoint_pickle, backup); print(f"overwrite_existing = True -> checkpoint moved to {backup}")
        else:
            records = pd.read_pickle(path_checkpoint_pickle).to_dict("records")
            all_results = [r for r in records if r.get("status") == "success" and r.get("prompt_version") == prompt_version]
            # token accounting from every response of this prompt_version that counted its frames right; responses that were excluded
            # earlier only because of a wrong self-reported count are kept when the accounting shows that every image arrived
            token_calibration = calibrate_frame_tokens([r for r in records if r.get("prompt_version") == prompt_version])
            if token_calibration:
                print("token accounting per model (text tokens a, tokens per image b): " + ", ".join(f"{k}: a={a:.0f}, b={b:.0f}" for k, (a, b) in token_calibration.items()))
            n_rescued = 0
            for r in records:
                if r.get("status") == "frame_count_mismatch" and r.get("prompt_version") == prompt_version and r.get("sympathy") is not None:
                    passed, how = frame_check_verdict(r.get("frames_reported"), r.get("frames_expected"), r.get("prompt_tokens"), r.get("model_key"), token_calibration)
                    if passed:
                        r["status"], r["frame_count_ok"], r["frame_check"] = "success", True, how; all_results.append(r); n_rescued += 1
            print(f"checkpoint loaded: {len(all_results)} successful responses kept ({n_rescued} accepted by the token accounting), "
                  f"{len(records) - len(all_results)} failed / stale will be retried")
    done = {result_key(r) for r in all_results}
    work = [(sn, mk) for sn, mk in tasks if f"{sn}_{mk}" not in done]
    print(f"tasks: {len(tasks)} (participants {len(participants)} x models {len(active_models)}); already done: {len(tasks) - len(work)}; calls to make: {len(work)}")
    def save_checkpoint():
        tmp = path_checkpoint_pickle + ".tmp"; pd.DataFrame(all_results).to_pickle(tmp); os.replace(tmp, path_checkpoint_pickle)
    def submit_in_phases(items, warmup):
        """Yield the futures as they complete. warmup = True: the first item of every model is submitted in a first phase (one call per
        model, the models in parallel) and the rest only after that phase has completed, so that the prompt of every model is already in
        the provider's cache when the parallel calls start. warmup = False: one phase."""
        if warmup:
            first, rest, seen = [], [], set()
            for sn, mk in items:
                (rest if mk in seen else first).append((sn, mk)); seen.add(mk)
            phases = [p for p in (first, rest) if p]
            if len(phases) == 2:
                print(f"[Cache warm-up] {len(first)} calls (one per model) go first; the other {len(rest)} follow in parallel")
        else:
            phases = [items]
        for phase in phases:
            futures = {executor.submit(rate_one, sn, mk): (sn, mk) for sn, mk in phase}
            for fut in as_completed(futures):
                yield fut
    executor = ThreadPoolExecutor(max_workers=max_workers)
    try:
        for fut in submit_in_phases(work, cache_warmup):
            all_results.append(fut.result()); save_checkpoint()
    except BaseException as e:
        print(f"\n[INTERRUPTED] {type(e).__name__}: {len(all_results)} responses are saved; re-run this cell to continue.")
        executor.shutdown(wait=False, cancel_futures=True); raise
    finally:
        save_checkpoint(); executor.shutdown(wait=True)
    df_results_15 = pd.DataFrame(all_results)
else:
    df_results_15 = experiment_helper_15.get_df_experiment_results() if os.path.exists(path_checkpoint_pickle) else pd.DataFrame()   # as Experiment 1 loads its results
    print(f"run_experiments = False: {len(df_results_15)} responses loaded from the checkpoint")

df_results_15 = df_results_15[df_results_15["Participant SN"].isin(set(participants)) & df_results_15["model_key"].isin(active_models)].reset_index(drop=True) if len(df_results_15) else df_results_15
if len(df_results_15):
    print("\nStatus per model:"); display(df_results_15.pivot_table(index="model_key", columns="status", values="Participant SN", aggfunc="count").fillna(0).astype(int))
    df_results_15.to_csv(path_output_csv, index=False, encoding="utf-8-sig")
    # one CSV per model in the column layout of experiment_1.csv (for merge_and_show_accuracy and for the other notebooks)
    df_success = df_results_15[df_results_15["status"] == "success"]
    df_by_model = {mk: d.drop(columns=["model_key"]).reset_index(drop=True) for mk, d in df_success.groupby("model_key")}
    for mk, d in df_by_model.items():
        d.to_csv(os.path.join(os.getcwd(), f"experiment_{exp_no_active}_{mk}.csv"), index=False, encoding="utf-8-sig")
    print(f"written: {path_output_csv} and experiment_15_<model>.csv for {list(df_by_model)}")

## Print Out Results 1 — every model on its own (as in Experiment 1)

In [ ]:
# === 1. Every model on its own: the Experiment 1 evaluation (threshold on sympathy / inspiration -> four groups, confusion matrix) ===
threshold_type = "mean"                  # the mean of each dimension's scores as the threshold (Experiment 1 of experiments_1_9 also reports this variant)
df_pred_by_model = {}                    # model -> the evaluated frame (Participant SN, GT_Group, Pred_Group, ...) for the joint predictions below
for mk, d in df_by_model.items():
    if len(d.dropna(subset=["Group"])) < 4:
        print(f"{mk}: too few rated participants with a group ({len(d)})"); continue
    print("\n" + "#" * 100 + f"\n{mk} -> {llm_models[mk].model_name}: {len(d)} participants\n" + "#" * 100)
    with pd.option_context("display.max_colwidth", 60):
        display(d[["Group Name", "Participant SN", "sympathy", "inspiration"]].head(10))
    df_pred_by_model[mk] = data_analysis_helper.merge_and_show_accuracy(df_list=[d], threshold_type=threshold_type, chart_helper=chart_helper,
                                                                        title=f"{exp_label}. Frames — {mk}", font_size=20)


## Print Out Results 2 — joint predictions: majority vote of the raters, then averaged scores

In [ ]:
# === 2. Joint predictions of the models: first the majority VOTE of the groups predicted by the individual models, then the AVERAGE of their scores ===
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
group_labels = [group_names[i] for i in (1, 2, 3, 4)]
if len(df_pred_by_model) >= 2:
    # ---- (a) vote: every model's Pred_Group is the one of its own evaluation above (its own thresholds); a participant's joint group is the
    #      most frequent of them; a tie goes to the group predicted by the model with the highest individual accuracy
    preds = pd.DataFrame({mk: d.set_index("Participant SN")["Pred_Group"] for mk, d in df_pred_by_model.items()})
    truth = pd.concat([d.set_index("Participant SN")["GT_Group"] for d in df_pred_by_model.values()]).groupby(level=0).first()
    acc_by_model = {mk: float((d["Pred_Group"] == d["GT_Group"]).mean()) for mk, d in df_pred_by_model.items()}
    best = max(acc_by_model, key=acc_by_model.get)
    def majority(row):
        v = row.dropna().astype(int)
        if not len(v):
            return np.nan
        counts = v.value_counts(); top = counts[counts == counts.max()].index.tolist()
        if len(top) == 1:
            return top[0]
        return int(row[best]) if pd.notna(row.get(best)) and int(row[best]) in top else top[0]
    df_vote = pd.DataFrame({"GT_Group": truth, "Pred_Group": preds.apply(majority, axis=1)}).dropna()
    y_true, y_pred = df_vote["GT_Group"].astype(int), df_vote["Pred_Group"].astype(int)
    acc, f1 = accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")
    print("\n" + "#" * 100 + f"\nJoint prediction (a): majority vote of {preds.shape[1]} raters ({', '.join(preds.columns)}); ties -> {best} (best individual accuracy)\n" + "#" * 100)
    print("Accuracy:", acc); print("Macro F1:", f1)
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4]); row_sums = cm.sum(axis=1, keepdims=True)
    per_class = np.diag(np.divide(cm, row_sums, where=row_sums != 0))
    for i, a in enumerate(per_class):
        print(f"Class {group_labels[i]} accuracy: {a:.2%}")
    print("\nExcel (row):\n" + "\t".join(f"{a:.4f}" for a in per_class))
    chart_helper.plot_confusion_matrix(cm, ticklabels=group_labels, show_percentage=True, task_name="",
                                       title=f"{exp_label}. Frames — joint vote of {preds.shape[1]} raters (Acc:{acc:.2f}, F1:{f1:.2f})",
                                       xticklabels=group_labels, yticklabels=group_labels, font_size=20)
    # ---- (b) average: merge_and_show_accuracy averages the scores of every frame it receives (equal weights) and thresholds the average.
    #      It merges the frames on Participant SN and renames only sympathy / inspiration / Group, so each rater contributes just those columns.
    print("\n" + "#" * 100 + f"\nJoint prediction (b): averaged scores of {len(df_by_model)} raters\n" + "#" * 100)
    dfs = [d[["Participant SN", "Group", "sympathy", "inspiration"]] for d in df_by_model.values()]
    df_joint = data_analysis_helper.merge_and_show_accuracy(df_list=dfs, threshold_type=threshold_type, chart_helper=chart_helper,
                                                            title=f"{exp_label}. Frames — joint average of {len(dfs)} raters", font_size=20)
else:
    print("joint predictions need at least two models with results")


## Inter-rater agreement of the six models

In [ ]:
# === Agreement between the models as raters: sympathy and inspiration (interval) and the predicted group (nominal) ===
import itertools
from scipy import stats

def icc_twoway(matrix):
    m = np.asarray(matrix, dtype=float); m = m[~np.isnan(m).any(axis=1)]; n, k = m.shape
    out = dict(icc21=np.nan, icc2k=np.nan, icc31=np.nan, n=n, k=k)
    if n < 3 or k < 2: return out
    grand = m.mean(); ms_r = k * np.sum((m.mean(axis=1) - grand) ** 2) / (n - 1); ms_c = n * np.sum((m.mean(axis=0) - grand) ** 2) / (k - 1)
    ms_e = np.sum((m - m.mean(axis=1, keepdims=True) - m.mean(axis=0, keepdims=True) + grand) ** 2) / ((n - 1) * (k - 1))
    d21 = ms_r + (k - 1) * ms_e + k * (ms_c - ms_e) / n; d2k = ms_r + (ms_c - ms_e) / n; d31 = ms_r + (k - 1) * ms_e
    out.update(icc21=(ms_r - ms_e) / d21 if d21 else np.nan, icc2k=(ms_r - ms_e) / d2k if d2k else np.nan, icc31=(ms_r - ms_e) / d31 if d31 else np.nan)
    return out

def krippendorff_alpha_interval(matrix):
    m = np.asarray(matrix, dtype=float); units = [r[~np.isnan(r)] for r in m]; units = [u for u in units if len(u) >= 2]
    if len(units) < 2: return np.nan
    n_values = sum(len(u) for u in units)
    d_o = sum(sum((a - b) ** 2 for a, b in itertools.permutations(u, 2)) / (len(u) - 1) for u in units) / n_values
    pooled = np.concatenate(units); d_e = np.sum((pooled[:, None] - pooled[None, :]) ** 2) / (n_values * (n_values - 1))
    return np.nan if d_e == 0 else 1 - d_o / d_e

def kendall_w(matrix):
    m = np.asarray(matrix, dtype=float); m = m[~np.isnan(m).any(axis=1)]; n, k = m.shape
    if n < 3 or k < 2: return np.nan
    ranks = np.column_stack([stats.rankdata(m[:, j]) for j in range(k)]); r_sum = ranks.sum(axis=1); s = np.sum((r_sum - r_sum.mean()) ** 2)
    ties = sum(np.sum(c ** 3 - c) for j in range(k) for c in [np.unique(m[:, j], return_counts=True)[1]])
    denom = k ** 2 * (n ** 3 - n) - k * ties
    return np.nan if denom <= 0 else 12 * s / denom

def fleiss_kappa(rows, categories):
    rows = [r for r in rows if all(pd.notna(x) for x in r)]
    if len(rows) < 2 or len(rows[0]) < 2: return np.nan
    k = len(rows[0]); counts = np.array([[sum(1 for x in r if x == c) for c in categories] for r in rows], dtype=float); n = len(rows)
    p_j = counts.sum(axis=0) / (n * k); p_i = (np.sum(counts ** 2, axis=1) - k) / (k * (k - 1)); p_e = np.sum(p_j ** 2)
    return np.nan if p_e == 1 else (p_i.mean() - p_e) / (1 - p_e)

def read_icc(v):
    return "n/a" if pd.isna(v) else "poor (< .50)" if v < 0.5 else "moderate (.50-.75)" if v < 0.75 else "good (.75-.90)" if v < 0.9 else "excellent (> .90)"

models_with_results = list(df_by_model)
if len(models_with_results) >= 2:
    wide = {dim: pd.DataFrame({mk: d.set_index("Participant SN")[dim].astype(float) for mk, d in df_by_model.items()}) for dim in ["sympathy", "inspiration"]}
    rows = {}
    for dim, w in wide.items():
        c = w.dropna(); icc = icc_twoway(c.to_numpy())
        pair_r = [stats.pearsonr(c[a], c[b])[0] for a, b in itertools.combinations(c.columns, 2) if len(c) >= 3 and c[a].std() > 0 and c[b].std() > 0]
        rows[dim] = {"n participants (rated by all)": len(c), "k raters": w.shape[1], "ICC(2,1) single rater": icc["icc21"], "ICC(2,k) mean of raters": icc["icc2k"],
                     "ICC(3,1) consistency": icc["icc31"], "Krippendorff alpha (interval)": krippendorff_alpha_interval(w.to_numpy()),
                     "Kendall W": kendall_w(c.to_numpy()), "mean pairwise Pearson r": np.mean(pair_r) if pair_r else np.nan,
                     "mean pairwise |difference|": np.mean([np.mean(np.abs(c[a] - c[b])) for a, b in itertools.combinations(c.columns, 2)]) if len(c) else np.nan}
    # predicted group per model with the same rule as merge_and_show_accuracy (threshold = mean over every model's scores)
    all_sym = pd.concat([w for w in [wide["sympathy"]]], axis=1).stack(); all_insp = wide["inspiration"].stack()
    t_sym, t_insp = all_sym.mean(), all_insp.mean()
    def classify(s, i):
        if pd.isna(s) or pd.isna(i): return np.nan
        return 1 if (s > t_sym and i > t_insp) else 2 if (s > t_sym) else 3 if (i > t_insp) else 4
    groups = pd.DataFrame({mk: [classify(s, i) for s, i in zip(wide["sympathy"][mk], wide["inspiration"][mk])] for mk in models_with_results}, index=wide["sympathy"].index).dropna()
    rows["predicted group"] = {"n participants (rated by all)": len(groups), "k raters": groups.shape[1], "Fleiss kappa": fleiss_kappa(groups.to_numpy().tolist(), [1, 2, 3, 4]),
                               "exact agreement (all raters same group)": float((groups.nunique(axis=1) == 1).mean()) if len(groups) else np.nan}
    df_agreement = pd.DataFrame(rows)
    with pd.option_context("display.precision", 3, "display.width", 200):
        print("=== Agreement of the raters (thresholds for the group: sympathy mean = %.2f, inspiration mean = %.2f) ===" % (t_sym, t_insp))
        display(df_agreement)
        for dim in ["sympathy", "inspiration"]:
            print(f"\n--- pairwise Pearson r of {dim} ---"); display(wide[dim].dropna().corr().round(3))
        for dim in ["sympathy", "inspiration"]:
            a = rows[dim]; print(f"{dim}: ICC(2,1) = {a['ICC(2,1) single rater']:.3f} -> {read_icc(a['ICC(2,1) single rater'])}; ICC(2,k) = {a['ICC(2,k) mean of raters']:.3f} -> {read_icc(a['ICC(2,k) mean of raters'])}; "
                  f"Krippendorff alpha = {a['Krippendorff alpha (interval)']:.3f}; Kendall W = {a['Kendall W']:.3f}")
    df_agreement.to_csv(os.path.join(os.getcwd(), "experiment_15_agreement.csv"), encoding="utf-8-sig")
else:
    print("inter-rater agreement needs at least two models with results")

## Raw responses of every model

In [ ]:
# === Every response as the model returned it (printed for the selection; all of them written to a Markdown file) ===
raw_show_participants = None   # None = all; or e.g. ["001", "002"]
raw_show_models = None         # None = all; or e.g. ["gpt"]
raw_max_print = 10
sel = df_results_15.copy() if len(df_results_15) else pd.DataFrame()
if len(sel):
    if raw_show_participants: sel = sel[sel["Participant SN"].isin(raw_show_participants)]
    if raw_show_models: sel = sel[sel["model_key"].isin(raw_show_models)]
    sel = sel.sort_values(["Participant SN", "model_key"])
    def block(r):
        return (f"### Participant {r['Participant SN']} ({r['Group Name']}) | model = {r['model_key']} ({r['model']}) | status = {r['status']} | "
                f"(sympathy, inspiration) = ({r['sympathy']}, {r['inspiration']}) | frames = {r['frames_reported']} / {r['frames_expected']} | tokens = {r['prompt_tokens']} (cached {r.get('cached_tokens')}) / {r['output_tokens']} | frame check = {r.get('frame_check', '')}\n\n"
                f"```\n{str(r['LLM_Result']).strip() or '(empty)'}\n```\n" + (f"\nerror:\n```\n{str(r['error'])[-1200:]}\n```\n" if str(r.get('error') or '').strip() else ""))
    path_raw_md = os.path.join(path_paper, "temp", f"experiment_{exp_no_active}_raw_responses.md")
    with open(path_raw_md, "w", encoding="utf-8") as f:
        f.write(f"# Experiment {exp_no_active} raw responses ({len(df_results_15)} responses)\n\n")
        for _, r in df_results_15.sort_values(["Participant SN", "model_key"]).iterrows():
            f.write(block(r) + "\n")
    print(f"{len(df_results_15)} raw responses written to {path_raw_md}\nPrinting {min(len(sel), raw_max_print)} of {len(sel)} selected:\n")
    for _, r in sel.head(raw_max_print).iterrows():
        print(block(r))

## Prompt-cache statistics

In [ ]:
# === Prompt-cache statistics: how much of the input was served from the providers' caches ===
# cached_tokens comes from the usage object of every response (OpenAI / DashScope prompt_tokens_details.cached_tokens, DeepSeek
# prompt_cache_hit_tokens, Gemini cached_content_token_count, Anthropic cache_read_input_tokens); the run log also prints it per call.
# The prompt of this experiment is short (a few hundred tokens), so providers with a minimum cacheable length (OpenAI 1,024 tokens, Anthropic
# 1,024-4,096 depending on the model) cannot cache it; the frames are participant-specific and are never cached.
df_cache = pd.read_pickle(path_checkpoint_pickle) if os.path.exists(path_checkpoint_pickle) else pd.DataFrame()
if len(df_cache) and "cached_tokens" in df_cache.columns:
    d = df_cache[df_cache["prompt_version"] == prompt_version].copy()
    for col in ("cached_tokens", "prompt_tokens"):
        d[col] = pd.to_numeric(d[col], errors="coerce")
    d["hit"] = d["cached_tokens"] > 0
    g = d.groupby("model_key").agg(n=("Participant SN", "size"), prompt_tokens=("prompt_tokens", "mean"), cached_tokens=("cached_tokens", "mean"),
                                   hit_share=("hit", "mean"), not_reported=("cached_tokens", lambda s: s.isna().mean()))
    g["cached_fraction"] = g["cached_tokens"] / g["prompt_tokens"]
    display(g.round(2))
    print("hit_share = share of responses with cached_tokens > 0 (the first call of every model is expected to miss); cached_fraction = mean cached / "
          "mean prompt tokens; not_reported = share whose provider returned no cache field (None).")
else:
    print("No responses with a cached_tokens column yet (run the experiment cell first).")


# Experiment 18 — six raters, still frames, step-23 recordings (video table `merged_video_23.plk`)

In [ ]:
select_experiment(18)


## Preview: frames and the complete prompt of one participant

In [ ]:
# === Preview ===
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
sample_sn = participants[0]
sample_video = df_participants.set_index("Participant SN").loc[sample_sn, "Compressed Video Path"]
sample_frames = extract_frames(sample_video, sample_sn)
print(f"Participant {sample_sn} ({df_participants.set_index('Participant SN').loc[sample_sn, 'Group Name']}): {len(sample_frames)} frames from {os.path.basename(sample_video)}")
n_cols = 8; n_rows = max(1, -(-len(sample_frames) // n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 1.6 * n_rows), squeeze=False)
for k, fr in enumerate(sample_frames):
    ax = axes[k // n_cols, k % n_cols]; ax.imshow(mpimg.imread(fr["path"])); ax.set_title(time_label(fr).replace("Frame at ", "").rstrip(":").split(" of")[0], fontsize=7); ax.axis("off")
for j in range(len(sample_frames), n_rows * n_cols):
    axes[j // n_cols, j % n_cols].axis("off")
plt.tight_layout(); plt.show()
print("\n" + "=" * 100 + f"\nFULL PROMPT (identical for every participant and every model; the frames follow it)\n" + "=" * 100)
print(template_main_15)

## Run: every participant × every model (checkpointed; re-run the cell to resume)

In [ ]:
# === Run (the experiment activated by select_experiment) ===
def make_empty_result(sn, model_key):
    row = df_participants.set_index("Participant SN").loc[sn]
    return {"Participant SN": sn, "Experiment Step": experiment_step, "Group": row["Group"], "Group Name": row["Group Name"],
            "Video Path": row["Video Path"], "Compressed Video Path": row["Compressed Video Path"], "model_key": model_key,
            "model": llm_models[model_key].model_name, "temperature": llm_temperature, "prompt_version": prompt_version,
            "frame_positions": str(frame_positions), "frames_expected": None, "frames_reported": None, "frame_count_ok": None,
            "prompt_tokens": None, "cached_tokens": None, "output_tokens": None, "finish_reason": "", "retries": 0, "frame_check": "",
            "sympathy": None, "inspiration": None, "explanation": "", "LLM_Result": "", "individual_prompt": template_main_15,
            "status": "pending", "error": ""}

def rate_one(sn, model_key):
    r = make_empty_result(sn, model_key)
    try:
        frames = extract_frames(r["Compressed Video Path"], sn)
        r["frames_expected"] = len(frames)
        t0 = time.time()
        text, ptok, otok, finish, ctok = generate_with_frames(model_key, template_main_15, frames)
        parsed = parse_llm_response(text)
        n_retries = 0
        while parsed["sympathy"] is None and n_retries < max_parse_retries:
            n_retries += 1
            print(f"  [RETRY {n_retries}] {sn} - {model_key}: no scores in the answer -> asking again with the reminder")
            text, ptok, otok, finish, ctok = generate_with_frames(model_key, template_main_15, frames, trailer_extra=parse_retry_reminder)
            parsed = parse_llm_response(text)
        passed, how = frame_check_verdict(parsed["frames_reported"], len(frames), ptok, model_key, token_calibration)
        r.update(retries=n_retries, LLM_Result=text, prompt_tokens=ptok, cached_tokens=ctok, output_tokens=otok, finish_reason=finish, sympathy=parsed["sympathy"],
                 inspiration=parsed["inspiration"], explanation=parsed["explanation"], frames_reported=parsed["frames_reported"],
                 frame_count_ok=passed, frame_check=how)
        if parsed["sympathy"] is None:
            r["status"] = "parse_failed"
        elif not passed:
            r["status"] = "frame_count_mismatch"
        else:
            r["status"] = "success"
        print(f"  [{'OK' if r['status'] == 'success' else r['status'].upper()}] {sn} ({r['Group Name']}) - {model_key}: (sympathy, inspiration) = "
              f"({parsed['sympathy']}, {parsed['inspiration']}), frames_analyzed={parsed['frames_reported']} (sent {len(frames)}, verified by {how}), "
              f"tokens={ptok} (cached {ctok}), {time.time() - t0:.0f}s")
    except Exception as e:
        r["status"], r["error"] = "error", traceback.format_exc()
        print(f"  [ERROR] {sn} - {model_key}: {e}")
    return r

def result_key(r):
    return f"{r['Participant SN']}_{r['model_key']}"

tasks = [(sn, mk) for sn in participants for mk in active_models]
if run_experiments:
    all_results = []
    if os.path.exists(path_checkpoint_pickle):
        if overwrite_existing:
            backup = path_checkpoint_pickle.replace(".plk", f"_backup_{time.strftime('%Y%m%d_%H%M%S')}.plk")
            os.replace(path_checkpoint_pickle, backup); print(f"overwrite_existing = True -> checkpoint moved to {backup}")
        else:
            records = pd.read_pickle(path_checkpoint_pickle).to_dict("records")
            all_results = [r for r in records if r.get("status") == "success" and r.get("prompt_version") == prompt_version]
            # token accounting from every response of this prompt_version that counted its frames right; responses that were excluded
            # earlier only because of a wrong self-reported count are kept when the accounting shows that every image arrived
            token_calibration = calibrate_frame_tokens([r for r in records if r.get("prompt_version") == prompt_version])
            if token_calibration:
                print("token accounting per model (text tokens a, tokens per image b): " + ", ".join(f"{k}: a={a:.0f}, b={b:.0f}" for k, (a, b) in token_calibration.items()))
            n_rescued = 0
            for r in records:
                if r.get("status") == "frame_count_mismatch" and r.get("prompt_version") == prompt_version and r.get("sympathy") is not None:
                    passed, how = frame_check_verdict(r.get("frames_reported"), r.get("frames_expected"), r.get("prompt_tokens"), r.get("model_key"), token_calibration)
                    if passed:
                        r["status"], r["frame_count_ok"], r["frame_check"] = "success", True, how; all_results.append(r); n_rescued += 1
            print(f"checkpoint loaded: {len(all_results)} successful responses kept ({n_rescued} accepted by the token accounting), "
                  f"{len(records) - len(all_results)} failed / stale will be retried")
    done = {result_key(r) for r in all_results}
    work = [(sn, mk) for sn, mk in tasks if f"{sn}_{mk}" not in done]
    print(f"tasks: {len(tasks)} (participants {len(participants)} x models {len(active_models)}); already done: {len(tasks) - len(work)}; calls to make: {len(work)}")
    def save_checkpoint():
        tmp = path_checkpoint_pickle + ".tmp"; pd.DataFrame(all_results).to_pickle(tmp); os.replace(tmp, path_checkpoint_pickle)
    def submit_in_phases(items, warmup):
        """Yield the futures as they complete. warmup = True: the first item of every model is submitted in a first phase (one call per
        model, the models in parallel) and the rest only after that phase has completed, so that the prompt of every model is already in
        the provider's cache when the parallel calls start. warmup = False: one phase."""
        if warmup:
            first, rest, seen = [], [], set()
            for sn, mk in items:
                (rest if mk in seen else first).append((sn, mk)); seen.add(mk)
            phases = [p for p in (first, rest) if p]
            if len(phases) == 2:
                print(f"[Cache warm-up] {len(first)} calls (one per model) go first; the other {len(rest)} follow in parallel")
        else:
            phases = [items]
        for phase in phases:
            futures = {executor.submit(rate_one, sn, mk): (sn, mk) for sn, mk in phase}
            for fut in as_completed(futures):
                yield fut
    executor = ThreadPoolExecutor(max_workers=max_workers)
    try:
        for fut in submit_in_phases(work, cache_warmup):
            all_results.append(fut.result()); save_checkpoint()
    except BaseException as e:
        print(f"\n[INTERRUPTED] {type(e).__name__}: {len(all_results)} responses are saved; re-run this cell to continue.")
        executor.shutdown(wait=False, cancel_futures=True); raise
    finally:
        save_checkpoint(); executor.shutdown(wait=True)
    df_results_15 = pd.DataFrame(all_results)
else:
    df_results_15 = experiment_helper_15.get_df_experiment_results() if os.path.exists(path_checkpoint_pickle) else pd.DataFrame()   # as Experiment 1 loads its results
    print(f"run_experiments = False: {len(df_results_15)} responses loaded from the checkpoint")

df_results_15 = df_results_15[df_results_15["Participant SN"].isin(set(participants)) & df_results_15["model_key"].isin(active_models)].reset_index(drop=True) if len(df_results_15) else df_results_15
if len(df_results_15):
    print("\nStatus per model:"); display(df_results_15.pivot_table(index="model_key", columns="status", values="Participant SN", aggfunc="count").fillna(0).astype(int))
    df_results_15.to_csv(path_output_csv, index=False, encoding="utf-8-sig")
    # one CSV per model in the column layout of experiment_1.csv (for merge_and_show_accuracy and for the other notebooks)
    df_success = df_results_15[df_results_15["status"] == "success"]
    df_by_model = {mk: d.drop(columns=["model_key"]).reset_index(drop=True) for mk, d in df_success.groupby("model_key")}
    for mk, d in df_by_model.items():
        d.to_csv(os.path.join(os.getcwd(), f"experiment_{exp_no_active}_{mk}.csv"), index=False, encoding="utf-8-sig")
    print(f"written: {path_output_csv} and experiment_15_<model>.csv for {list(df_by_model)}")

## Print Out Results 1 — every model on its own (as in Experiment 1)

In [ ]:
# === 1. Every model on its own: the Experiment 1 evaluation (threshold on sympathy / inspiration -> four groups, confusion matrix) ===
threshold_type = "mean"                  # the mean of each dimension's scores as the threshold (Experiment 1 of experiments_1_9 also reports this variant)
df_pred_by_model = {}                    # model -> the evaluated frame (Participant SN, GT_Group, Pred_Group, ...) for the joint predictions below
for mk, d in df_by_model.items():
    if len(d.dropna(subset=["Group"])) < 4:
        print(f"{mk}: too few rated participants with a group ({len(d)})"); continue
    print("\n" + "#" * 100 + f"\n{mk} -> {llm_models[mk].model_name}: {len(d)} participants\n" + "#" * 100)
    with pd.option_context("display.max_colwidth", 60):
        display(d[["Group Name", "Participant SN", "sympathy", "inspiration"]].head(10))
    df_pred_by_model[mk] = data_analysis_helper.merge_and_show_accuracy(df_list=[d], threshold_type=threshold_type, chart_helper=chart_helper,
                                                                        title=f"{exp_label}. Frames — {mk}", font_size=20)


## Print Out Results 2 — joint predictions: majority vote of the raters, then averaged scores

In [ ]:
# === 2. Joint predictions of the models: first the majority VOTE of the groups predicted by the individual models, then the AVERAGE of their scores ===
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
group_labels = [group_names[i] for i in (1, 2, 3, 4)]
if len(df_pred_by_model) >= 2:
    # ---- (a) vote: every model's Pred_Group is the one of its own evaluation above (its own thresholds); a participant's joint group is the
    #      most frequent of them; a tie goes to the group predicted by the model with the highest individual accuracy
    preds = pd.DataFrame({mk: d.set_index("Participant SN")["Pred_Group"] for mk, d in df_pred_by_model.items()})
    truth = pd.concat([d.set_index("Participant SN")["GT_Group"] for d in df_pred_by_model.values()]).groupby(level=0).first()
    acc_by_model = {mk: float((d["Pred_Group"] == d["GT_Group"]).mean()) for mk, d in df_pred_by_model.items()}
    best = max(acc_by_model, key=acc_by_model.get)
    def majority(row):
        v = row.dropna().astype(int)
        if not len(v):
            return np.nan
        counts = v.value_counts(); top = counts[counts == counts.max()].index.tolist()
        if len(top) == 1:
            return top[0]
        return int(row[best]) if pd.notna(row.get(best)) and int(row[best]) in top else top[0]
    df_vote = pd.DataFrame({"GT_Group": truth, "Pred_Group": preds.apply(majority, axis=1)}).dropna()
    y_true, y_pred = df_vote["GT_Group"].astype(int), df_vote["Pred_Group"].astype(int)
    acc, f1 = accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")
    print("\n" + "#" * 100 + f"\nJoint prediction (a): majority vote of {preds.shape[1]} raters ({', '.join(preds.columns)}); ties -> {best} (best individual accuracy)\n" + "#" * 100)
    print("Accuracy:", acc); print("Macro F1:", f1)
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4]); row_sums = cm.sum(axis=1, keepdims=True)
    per_class = np.diag(np.divide(cm, row_sums, where=row_sums != 0))
    for i, a in enumerate(per_class):
        print(f"Class {group_labels[i]} accuracy: {a:.2%}")
    print("\nExcel (row):\n" + "\t".join(f"{a:.4f}" for a in per_class))
    chart_helper.plot_confusion_matrix(cm, ticklabels=group_labels, show_percentage=True, task_name="",
                                       title=f"{exp_label}. Frames — joint vote of {preds.shape[1]} raters (Acc:{acc:.2f}, F1:{f1:.2f})",
                                       xticklabels=group_labels, yticklabels=group_labels, font_size=20)
    # ---- (b) average: merge_and_show_accuracy averages the scores of every frame it receives (equal weights) and thresholds the average.
    #      It merges the frames on Participant SN and renames only sympathy / inspiration / Group, so each rater contributes just those columns.
    print("\n" + "#" * 100 + f"\nJoint prediction (b): averaged scores of {len(df_by_model)} raters\n" + "#" * 100)
    dfs = [d[["Participant SN", "Group", "sympathy", "inspiration"]] for d in df_by_model.values()]
    df_joint = data_analysis_helper.merge_and_show_accuracy(df_list=dfs, threshold_type=threshold_type, chart_helper=chart_helper,
                                                            title=f"{exp_label}. Frames — joint average of {len(dfs)} raters", font_size=20)
else:
    print("joint predictions need at least two models with results")


## Inter-rater agreement of the six models

In [ ]:
# === Agreement between the models as raters: sympathy and inspiration (interval) and the predicted group (nominal) ===
import itertools
from scipy import stats

def icc_twoway(matrix):
    m = np.asarray(matrix, dtype=float); m = m[~np.isnan(m).any(axis=1)]; n, k = m.shape
    out = dict(icc21=np.nan, icc2k=np.nan, icc31=np.nan, n=n, k=k)
    if n < 3 or k < 2: return out
    grand = m.mean(); ms_r = k * np.sum((m.mean(axis=1) - grand) ** 2) / (n - 1); ms_c = n * np.sum((m.mean(axis=0) - grand) ** 2) / (k - 1)
    ms_e = np.sum((m - m.mean(axis=1, keepdims=True) - m.mean(axis=0, keepdims=True) + grand) ** 2) / ((n - 1) * (k - 1))
    d21 = ms_r + (k - 1) * ms_e + k * (ms_c - ms_e) / n; d2k = ms_r + (ms_c - ms_e) / n; d31 = ms_r + (k - 1) * ms_e
    out.update(icc21=(ms_r - ms_e) / d21 if d21 else np.nan, icc2k=(ms_r - ms_e) / d2k if d2k else np.nan, icc31=(ms_r - ms_e) / d31 if d31 else np.nan)
    return out

def krippendorff_alpha_interval(matrix):
    m = np.asarray(matrix, dtype=float); units = [r[~np.isnan(r)] for r in m]; units = [u for u in units if len(u) >= 2]
    if len(units) < 2: return np.nan
    n_values = sum(len(u) for u in units)
    d_o = sum(sum((a - b) ** 2 for a, b in itertools.permutations(u, 2)) / (len(u) - 1) for u in units) / n_values
    pooled = np.concatenate(units); d_e = np.sum((pooled[:, None] - pooled[None, :]) ** 2) / (n_values * (n_values - 1))
    return np.nan if d_e == 0 else 1 - d_o / d_e

def kendall_w(matrix):
    m = np.asarray(matrix, dtype=float); m = m[~np.isnan(m).any(axis=1)]; n, k = m.shape
    if n < 3 or k < 2: return np.nan
    ranks = np.column_stack([stats.rankdata(m[:, j]) for j in range(k)]); r_sum = ranks.sum(axis=1); s = np.sum((r_sum - r_sum.mean()) ** 2)
    ties = sum(np.sum(c ** 3 - c) for j in range(k) for c in [np.unique(m[:, j], return_counts=True)[1]])
    denom = k ** 2 * (n ** 3 - n) - k * ties
    return np.nan if denom <= 0 else 12 * s / denom

def fleiss_kappa(rows, categories):
    rows = [r for r in rows if all(pd.notna(x) for x in r)]
    if len(rows) < 2 or len(rows[0]) < 2: return np.nan
    k = len(rows[0]); counts = np.array([[sum(1 for x in r if x == c) for c in categories] for r in rows], dtype=float); n = len(rows)
    p_j = counts.sum(axis=0) / (n * k); p_i = (np.sum(counts ** 2, axis=1) - k) / (k * (k - 1)); p_e = np.sum(p_j ** 2)
    return np.nan if p_e == 1 else (p_i.mean() - p_e) / (1 - p_e)

def read_icc(v):
    return "n/a" if pd.isna(v) else "poor (< .50)" if v < 0.5 else "moderate (.50-.75)" if v < 0.75 else "good (.75-.90)" if v < 0.9 else "excellent (> .90)"

models_with_results = list(df_by_model)
if len(models_with_results) >= 2:
    wide = {dim: pd.DataFrame({mk: d.set_index("Participant SN")[dim].astype(float) for mk, d in df_by_model.items()}) for dim in ["sympathy", "inspiration"]}
    rows = {}
    for dim, w in wide.items():
        c = w.dropna(); icc = icc_twoway(c.to_numpy())
        pair_r = [stats.pearsonr(c[a], c[b])[0] for a, b in itertools.combinations(c.columns, 2) if len(c) >= 3 and c[a].std() > 0 and c[b].std() > 0]
        rows[dim] = {"n participants (rated by all)": len(c), "k raters": w.shape[1], "ICC(2,1) single rater": icc["icc21"], "ICC(2,k) mean of raters": icc["icc2k"],
                     "ICC(3,1) consistency": icc["icc31"], "Krippendorff alpha (interval)": krippendorff_alpha_interval(w.to_numpy()),
                     "Kendall W": kendall_w(c.to_numpy()), "mean pairwise Pearson r": np.mean(pair_r) if pair_r else np.nan,
                     "mean pairwise |difference|": np.mean([np.mean(np.abs(c[a] - c[b])) for a, b in itertools.combinations(c.columns, 2)]) if len(c) else np.nan}
    # predicted group per model with the same rule as merge_and_show_accuracy (threshold = mean over every model's scores)
    all_sym = pd.concat([w for w in [wide["sympathy"]]], axis=1).stack(); all_insp = wide["inspiration"].stack()
    t_sym, t_insp = all_sym.mean(), all_insp.mean()
    def classify(s, i):
        if pd.isna(s) or pd.isna(i): return np.nan
        return 1 if (s > t_sym and i > t_insp) else 2 if (s > t_sym) else 3 if (i > t_insp) else 4
    groups = pd.DataFrame({mk: [classify(s, i) for s, i in zip(wide["sympathy"][mk], wide["inspiration"][mk])] for mk in models_with_results}, index=wide["sympathy"].index).dropna()
    rows["predicted group"] = {"n participants (rated by all)": len(groups), "k raters": groups.shape[1], "Fleiss kappa": fleiss_kappa(groups.to_numpy().tolist(), [1, 2, 3, 4]),
                               "exact agreement (all raters same group)": float((groups.nunique(axis=1) == 1).mean()) if len(groups) else np.nan}
    df_agreement = pd.DataFrame(rows)
    with pd.option_context("display.precision", 3, "display.width", 200):
        print("=== Agreement of the raters (thresholds for the group: sympathy mean = %.2f, inspiration mean = %.2f) ===" % (t_sym, t_insp))
        display(df_agreement)
        for dim in ["sympathy", "inspiration"]:
            print(f"\n--- pairwise Pearson r of {dim} ---"); display(wide[dim].dropna().corr().round(3))
        for dim in ["sympathy", "inspiration"]:
            a = rows[dim]; print(f"{dim}: ICC(2,1) = {a['ICC(2,1) single rater']:.3f} -> {read_icc(a['ICC(2,1) single rater'])}; ICC(2,k) = {a['ICC(2,k) mean of raters']:.3f} -> {read_icc(a['ICC(2,k) mean of raters'])}; "
                  f"Krippendorff alpha = {a['Krippendorff alpha (interval)']:.3f}; Kendall W = {a['Kendall W']:.3f}")
    df_agreement.to_csv(os.path.join(os.getcwd(), "experiment_15_agreement.csv"), encoding="utf-8-sig")
else:
    print("inter-rater agreement needs at least two models with results")

## Raw responses of every model

In [ ]:
# === Every response as the model returned it (printed for the selection; all of them written to a Markdown file) ===
raw_show_participants = None   # None = all; or e.g. ["001", "002"]
raw_show_models = None         # None = all; or e.g. ["gpt"]
raw_max_print = 10
sel = df_results_15.copy() if len(df_results_15) else pd.DataFrame()
if len(sel):
    if raw_show_participants: sel = sel[sel["Participant SN"].isin(raw_show_participants)]
    if raw_show_models: sel = sel[sel["model_key"].isin(raw_show_models)]
    sel = sel.sort_values(["Participant SN", "model_key"])
    def block(r):
        return (f"### Participant {r['Participant SN']} ({r['Group Name']}) | model = {r['model_key']} ({r['model']}) | status = {r['status']} | "
                f"(sympathy, inspiration) = ({r['sympathy']}, {r['inspiration']}) | frames = {r['frames_reported']} / {r['frames_expected']} | tokens = {r['prompt_tokens']} (cached {r.get('cached_tokens')}) / {r['output_tokens']} | frame check = {r.get('frame_check', '')}\n\n"
                f"```\n{str(r['LLM_Result']).strip() or '(empty)'}\n```\n" + (f"\nerror:\n```\n{str(r['error'])[-1200:]}\n```\n" if str(r.get('error') or '').strip() else ""))
    path_raw_md = os.path.join(path_paper, "temp", f"experiment_{exp_no_active}_raw_responses.md")
    with open(path_raw_md, "w", encoding="utf-8") as f:
        f.write(f"# Experiment {exp_no_active} raw responses ({len(df_results_15)} responses)\n\n")
        for _, r in df_results_15.sort_values(["Participant SN", "model_key"]).iterrows():
            f.write(block(r) + "\n")
    print(f"{len(df_results_15)} raw responses written to {path_raw_md}\nPrinting {min(len(sel), raw_max_print)} of {len(sel)} selected:\n")
    for _, r in sel.head(raw_max_print).iterrows():
        print(block(r))

## Prompt-cache statistics

In [ ]:
# === Prompt-cache statistics: how much of the input was served from the providers' caches ===
# cached_tokens comes from the usage object of every response (OpenAI / DashScope prompt_tokens_details.cached_tokens, DeepSeek
# prompt_cache_hit_tokens, Gemini cached_content_token_count, Anthropic cache_read_input_tokens); the run log also prints it per call.
# The prompt of this experiment is short (a few hundred tokens), so providers with a minimum cacheable length (OpenAI 1,024 tokens, Anthropic
# 1,024-4,096 depending on the model) cannot cache it; the frames are participant-specific and are never cached.
df_cache = pd.read_pickle(path_checkpoint_pickle) if os.path.exists(path_checkpoint_pickle) else pd.DataFrame()
if len(df_cache) and "cached_tokens" in df_cache.columns:
    d = df_cache[df_cache["prompt_version"] == prompt_version].copy()
    for col in ("cached_tokens", "prompt_tokens"):
        d[col] = pd.to_numeric(d[col], errors="coerce")
    d["hit"] = d["cached_tokens"] > 0
    g = d.groupby("model_key").agg(n=("Participant SN", "size"), prompt_tokens=("prompt_tokens", "mean"), cached_tokens=("cached_tokens", "mean"),
                                   hit_share=("hit", "mean"), not_reported=("cached_tokens", lambda s: s.isna().mean()))
    g["cached_fraction"] = g["cached_tokens"] / g["prompt_tokens"]
    display(g.round(2))
    print("hit_share = share of responses with cached_tokens > 0 (the first call of every model is expected to miss); cached_fraction = mean cached / "
          "mean prompt tokens; not_reported = share whose provider returned no cache field (None).")
else:
    print("No responses with a cached_tokens column yet (run the experiment cell first).")


# Footnote

- Experiment 1 of experiments_1_9 (`ExperimentHelper.run_exp_moral_elevation`) uploads every compressed recording to the Gemini Files API. Experiments 17 and 18 send still frames inline instead, so every provider receives exactly the same images in the same layout; the prompt is Experiment 1's with the attachment described as frames; no frame count is requested.
- The evaluation is the one of Experiment 1 (`merge_and_show_accuracy`, mean thresholds on the sympathy / inspiration scores, four-group confusion matrix); the joint predictions are the majority vote of the raters' predicted groups and the same routine over their averaged scores; the inter-rater agreement is the addition of this notebook.
- Only this notebook was added; `configs.py` (now pointing `path_psychai` at `psychai_20290921`), the helpers and the other notebooks are otherwise unchanged.
